# Remote CLIP Colab Worker

Runs the **Remote CLIP Colab** inference worker on this runtime. Your local
ComfyUI connects with the printed `base_url` + `auth_token` via the
`Load Remote CLIP (Colab)` node, and manages the worker (status / load /
switch / unload models, LoRAs, cache, tunnel, shutdown) through the
`Remote CLIP Controller` node — no notebook UI needed.

**Cells**
1. Setup — fetch the runtime (git clone / uploaded folder / Google Drive) and install deps
2. Configure — token, tunnel, backend, startup models
3. Launch — start the worker in the background, wait for the public URL
4. Download — fetch text-encoder / LoRA files into `models/` (user-initiated)
5. Log — tail the worker log
6. Stop — shut the worker down

Runtime type: **GPU (T4/L4/A100)** recommended — the native backend (ComfyUI's own
text-encoder stack) activates automatically. On **TPU** the worker falls back to
the pure-transformers backend. NVFP4/AWQ checkpoints (e.g. MiniMax H3) need an
Ampere-or-newer GPU.


In [ ]:
# @title 1 · Setup runtime { display-mode: "form" }
import os, subprocess, sys

REPO_URL = ""  # @param {type:"string"} — set to clone; leave empty if the colab/ folder is already here or on Drive
RUNTIME_DIR = "/content/ComfyUI-RemoteCLIPColab/colab"  # @param {type:"string"} — e.g. /content/ComfyUI-RemoteCLIPColab/colab or a Drive path

if REPO_URL.strip() and not os.path.isdir(RUNTIME_DIR):
    repo_root = os.path.dirname(RUNTIME_DIR.rstrip("/")) or "/content/ComfyUI-RemoteCLIPColab"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL.strip(), repo_root], check=True)

assert os.path.isfile(os.path.join(RUNTIME_DIR, "worker.py")), (
    f"worker.py not found under {RUNTIME_DIR}.\n"
    "Set REPO_URL to your fork, or upload the colab/ folder (File → Upload), "
    "or mount Drive and point RUNTIME_DIR at it.")
os.chdir(RUNTIME_DIR)

def _is_tpu():
    try:
        import torch_xla  # noqa: F401
        return True
    except ImportError:
        return False

print("installing worker dependencies (first run takes a couple of minutes) ...")
if _is_tpu():
    # TPU images pin torch+torch_xla as a pair; reinstalling torch breaks XLA.
    pkgs = [l.split("#", 1)[0].strip() for l in open("requirements.txt")]
    pkgs = [p for p in pkgs if p and not p.startswith(("torch", "comfy-"))]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("TPU runtime: kept preinstalled torch/torch_xla, skipped comfy-kitchen/aimdo (native backend unused on TPU)")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # optional GPU inference-acceleration kernels for the native backend
    # (cell 2 ATTENTION=sage/flash); sdpa needs no extra package
    for pkg in ("sageattention",):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           capture_output=True, text=True)
        print(pkg, "installed" if r.returncode == 0 else "unavailable (ATTENTION=sage will be rejected)")
print("runtime ready:", os.getcwd())

In [ ]:
# @title 2 · Configure { display-mode: "form" }
AUTH_TOKEN = ""  # @param {type:"string"} — leave empty to auto-generate
TUNNEL = "cloudflare"  # @param ["cloudflare", "ngrok", "direct"]
NGROK_TOKEN = ""  # @param {type:"string"} — only for TUNNEL=ngrok
PORT = 8199  # @param {type:"integer"}
ENGINE = "auto"  # @param ["auto", "native", "hf"] — native = ComfyUI's own stack (recommended on GPU)
DEVICE = "auto"  # @param ["auto", "cuda", "tpu", "cpu"]
ATTENTION = "auto"  # @param ["auto", "sdpa", "sage", "flash"] — GPU inference acceleration
ATTENTION_HF = "sdpa"  # @param ["sdpa", "eager", "flash_attention_2"] — hf-backend kernel (CUDA)
XLA_CACHE = True  # @param {type:"boolean"} — TPU only: persist XLA compilation cache across restarts

import os
os.chdir(RUNTIME_DIR)
print("config ok — run cell 3 to launch")

In [ ]:
# @title 3 · Launch worker { display-mode: "form" }
import json, secrets, subprocess, sys, time, urllib.request

TOKEN = AUTH_TOKEN.strip() or secrets.token_hex(16)
open("worker_token.txt", "w").write(TOKEN)

args = [sys.executable, "worker.py", "--tunnel", TUNNEL, "--host", "127.0.0.1",
        "--port", str(PORT), "--token", TOKEN, "--engine", ENGINE, "--device", DEVICE,
        "--attention", ATTENTION, "--attention-hf", ATTENTION_HF]
try:
    import torch_xla  # noqa: F401 — TPU runtime
    if XLA_CACHE:
        args += ["--xla-cache", "/content/rcp_xla_cache"]
except ImportError:
    pass
if NGROK_TOKEN.strip():
    args += ["--ngrok-token", NGROK_TOKEN.strip()]

log_f = open("worker.log", "w")
WORKER = subprocess.Popen(args, stdout=log_f, stderr=subprocess.STDOUT)
print("worker pid:", WORKER.pid, "| log: worker.log")

def _api(path):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}",
                                 headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

def _fail():
    print(open("worker.log").read()[-3000:])
    raise SystemExit("worker failed to start — log tail above")

for i in range(90):
    time.sleep(2)
    if WORKER.poll() is not None:
        _fail()
    try:
        if _api("/health").get("ok"):
            print(f"worker up after {(i + 1) * 2}s")
            break
    except Exception:
        pass
else:
    _fail()

if TUNNEL != "direct":
    url = None
    for i in range(90):
        time.sleep(2)
        if WORKER.poll() is not None:
            _fail()
        try:
            url = _api("/v1/tunnel").get("public_url")
        except Exception:
            pass
        if url:
            break
    print("#" * 64)
    print("  CONNECT FROM THE LOCAL COMFYUI NODE WITH:")
    print(f"    base_url   : {url or 'TUNNEL NOT READY — check worker.log'}")
    print(f"    auth_token : {TOKEN}")
    print("#" * 64)
else:
    print(f"direct mode — expose port {PORT} yourself | token: {TOKEN}")
print("\nnext: cell 4 to download weights (optional); manage the worker from "
      "ComfyUI's Remote CLIP Controller node")

In [ ]:
# @title 4 · Download a model / LoRA { display-mode: "form" }
import os, re, subprocess

URL = ""  # @param {type:"string"} — direct file URL (e.g. an HF resolve link)
SUBDIR = "text_encoders"  # @param ["text_encoders", "clip", "loras", "embeddings", ""]

assert URL.strip(), "set a URL first"
dest = os.path.join(RUNTIME_DIR, "models", SUBDIR)
os.makedirs(dest, exist_ok=True)
name = os.path.basename(URL.split("?")[0]) or "download.safetensors"
path = os.path.join(dest, name)
subprocess.run(["wget", "-q", "--show-progress", "-O", path, URL.strip()], check=True)
rel = os.path.relpath(path, RUNTIME_DIR).replace("\\", "/")
size_gb = os.path.getsize(path) / 1e9
print(f"saved {rel} ({size_gb:.2f} GB)")
print(f"load it with kind like:  clip_l:{rel}   or from ComfyUI's "
      f"Remote CLIP Controller node (load_model)")

In [ ]:
# @title 5 · Worker log tail { display-mode: "form" }
LINES = 40  # @param {type:"integer"}
print("\n".join(open("worker.log").read().splitlines()[-LINES:]))

In [ ]:
# @title 6 · Stop worker { display-mode: "form" }
import subprocess
if "WORKER" in globals() and WORKER.poll() is None:
    WORKER.terminate()
    print("worker terminated (pid", WORKER.pid, ")")
else:
    subprocess.run(["pkill", "-f", "worker.py"])
    print("any stray workers killed")